# Calling cuPIQP with PyTorch Tensors

This notebook shows how to interop with PyTorch in cuPIQP. It mirrors [Getting started](getting_started.ipynb), but every input is a **CUDA `torch.Tensor`** instead of a cupy array. 
The cuPIQP solvers accept GPU torch tensors directly (zero-copy through the CUDA array interface / [DLPack](https://dmlc.github.io/dlpack/latest/)); the result arrays come back as cupy and can be returned to torch
with [`torch.from_dlpack`](https://docs.pytorch.org/docs/stable/generated/torch.from_dlpack.html), also zero-copy.

In [1]:
import torch
from cupiqp import DenseSolver, SparseSolver, Status

torch.set_default_dtype(torch.float64)
assert torch.cuda.is_available(), "cuPIQP requires a CUDA device."
device = torch.device("cuda")

## Example QPs

We consider the following problem:

$$
\begin{aligned}
\min_{x_1,\,x_2}\quad & \tfrac12\bigl(6 x_1^2 + 4 x_2^2\bigr) - x_1 - 4 x_2 \\
\text{s.t.}\quad
  & x_1 - 2 x_2 = 1, \\
  & x_1 - x_2 \leq 0.2, \\
  & 2x_1 \le -1, \\
  & \theta \le x_1 \le 1,
\end{aligned}
$$

where $\theta$ is the lower bound on $x_1$ which varies among problems in the batch. In
this example, we create a batch of $B=4$ problems. For simplicity, we assign different
values of $\theta$ to each problem while keeping other data the same:

| problem | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| $\theta$ | $-2.0$ | $-1.5$ | $-1.0$ | $2.0$ |

Problem 3 is infeasible since the bounds of $x_1$ become $2 \leq x_1 \leq 1$, so the box
is empty.

Notice that cuPIQP requires the **same finite-bound pattern**, i.e., the set of finite
bounds must match and only their values may change. This is also required when calling
`update()` in subsequent solves.

`solver.result.x` then has shape `(B, n)` and `solver.result.info.status` is a list of
$B$ statuses (one per problem) -- so the infeasible problem reports a different status
from the rest.

In [2]:
# --- Template data (CUDA torch tensors) ---
# quadratic + linear cost
P = torch.tensor([[6.0, 0.0],
                  [0.0, 4.0]], device=device)
c = torch.tensor([-1.0, -4.0], device=device)

# equality constraint:  A x = b
A = torch.tensor([[1.0, -2.0]], device=device)
b = torch.tensor([1.0], device=device)

# two-sided inequalities:  h_l <= G x <= h_u   (use -inf / +inf for one-sided)
G   = torch.tensor([[1.0, -1.0],
                    [2.0,  0.0]], device=device)
h_l = torch.tensor([-torch.inf, -torch.inf], device=device)
h_u = torch.tensor([0.2, -1.0], device=device)

# box bounds:  x_l <= x <= x_u
x_l = torch.tensor([-1.0, -torch.inf], device=device)
x_u = torch.tensor([ 1.0,  torch.inf], device=device)


# --- Build batched data ---
B = 4  # batch size

# replicate the shared matrices / vectors along the leading batch dimension -> (B, ...)
stack = lambda M: torch.stack([M] * B)
P_batch, c_batch, A_batch, b_batch, G_batch = stack(P), stack(c), stack(A), stack(b), stack(G)
h_l_batch, h_u_batch, x_u_batch = stack(h_l), stack(h_u), stack(x_u)
x_l_batch = torch.tensor([
    [-2.0, -torch.inf],
    [-1.5, -torch.inf],
    [-1.0, -torch.inf],
    [ 2.0, -torch.inf],  # infeasible!
], device=device)

## Dense solver

`DenseSolver` works with **dense** GPU arrays for `P`, `A`, `G`. We hand `setup` the
batched `(B, ...)` **torch** tensors directly, call `solve()`, then read the per-problem
solution and status off `solver.result`.

In [3]:
solver_dense = DenseSolver()
solver_dense.settings.verbose = True

solver_dense.setup(P=P_batch, c=c_batch, A=A_batch, b=b_batch, G=G_batch,
                   h_l=h_l_batch, h_u=h_u_batch, x_l=x_l_batch, x_u=x_u_batch)
status_dense = solver_dense.solve()  # list of length B, one Status per problem

----------------------------------------------------------
       cuPIQP v0.1.0 - GPU-accelerated PIQP solver        
                    (c) Fenglong Song                     
   Ecole Polytechnique Federale de Lausanne (EPFL) 2026   
----------------------------------------------------------
dense backend:
batch size B = 4
variables n = 2
equality constraints p = 1
inequality constraints m = 2
inequality lower bounds n_h_l = 0
inequality upper bounds n_h_u = 2
variable lower bounds n_x_l = 1
variable upper bounds n_x_u = 1

iter  solved       gap_max     p_res_max     d_res_max     rho_max   delta_max      mu_max  p_step  d_step
   0     0/4   1.90819e+01   2.69408e+00   4.84870e+00   1.000e-06   1.000e-04   1.652e+00  0.0000  0.0000
   1     0/4   2.16919e+03   2.52134e+00   4.41792e+00   1.000e-07   1.000e-05   2.362e+01  0.0648  0.9268
   2     0/4   1.48750e+06   2.44501e+00   1.44267e+00   5.000e-08   5.000e-06   1.140e+03  0.1477  0.1575
   3     0/4   3.76528e+06   2.45375e+00

### Extract the results

The result lives on `solver.result`:

- `solver.result.info.status`: per-problem status (a list of length `B`)
- `solver.result.x`: primal solution
- `solver.result.y`, `solver.result.z_l`, `solver.result.z_u`, `solver.result.z_bl`, `solver.result.z_bu`: dual solutions

The solver returns **cupy** arrays. Bring any of them into PyTorch with
`torch.from_dlpack(...)` -- a **zero-copy** view that stays on the GPU:

In [4]:
# result arrays are CuPy arrays; view them as torch tensors zero-copy on GPU
x_cupy = solver_dense.result.x

print(f"result.x is a {type(x_cupy)} object with shape {x_cupy.shape}")
print(f"cupy device pointer:  {x_cupy.data.ptr:x}")  # :x means hexadecimal format

print("\nIt can be converted to a torch.Tensor with zero-copy via DLPack")
x_torch = torch.from_dlpack(x_cupy)

print(f"Torch device pointer: {x_torch.data_ptr():x}")  # :x means hexadecimal format
print("\nSame device memory?", x_cupy.data.ptr == x_torch.data_ptr())

result.x is a <class 'cupy.ndarray'> object with shape (4, 2)
cupy device pointer:  70e647400200

It can be converted to a torch.Tensor with zero-copy via DLPack
Torch device pointer: 70e647400200

Same device memory? True


To copy a result to the CPU, convert it to torch and call `.cpu()` (or use cupy's
`.get()` directly on `solver.result.x`):

In [5]:
x_dense_host = torch.from_dlpack(solver_dense.result.x).cpu()  # (B, n) on CPU
print(f"x on host: {type(x_dense_host).__name__} {tuple(x_dense_host.shape)} on {x_dense_host.device}")

x on host: Tensor (4, 2) on cpu


## Sparse solver

`SparseSolver` takes `P`, `A`, `G` as sparse CSR matrices on the GPU. For PyTorch inputs, use a [`torch.sparse_csr_tensor`](https://docs.pytorch.org/docs/stable/generated/torch.sparse_csr_tensor.html) object to store the batched CSR matrices, with a restriction that every matrix in the batch must share **the same** `crow_indices` / `col_indices` pattern while the stored `values` may differ across the batch. 

The vectors `c, b, h_l, h_u, x_l, x_u` stay dense `(B, ...)` torch tensors, exactly as in the dense case.

In [6]:
P_csr_batch = P_batch.to_sparse_csr()
A_csr_batch = A_batch.to_sparse_csr()
G_csr_batch = G_batch.to_sparse_csr()

# --- Alternatively: take P as an example ---
# P_csr = P.to_sparse_csr()
# P_csr_batch = torch.sparse_csr_tensor(
#     crow_indices=torch.tile(P_csr.crow_indices(), (B, 1)),
#     col_indices=torch.tile(P_csr.col_indices(), (B, 1)),
#     values=torch.tile(P_csr.values(), (B, 1))
#     )

solver_sparse = SparseSolver()
solver_sparse.settings.verbose = True

solver_sparse.setup(
    P=P_csr_batch, c=c_batch,
    A=A_csr_batch, b=b_batch,
    G=G_csr_batch, h_l=h_l_batch, h_u=h_u_batch,
    x_l=x_l_batch, x_u=x_u_batch,
)
status_sparse = solver_sparse.solve()

/tmp/ipykernel_3432840/2761483166.py:1: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  P_csr_batch = P_batch.to_sparse_csr()


----------------------------------------------------------
       cuPIQP v0.1.0 - GPU-accelerated PIQP solver        
                    (c) Fenglong Song                     
   Ecole Polytechnique Federale de Lausanne (EPFL) 2026   
----------------------------------------------------------
sparse backend:
batch size B = 4
variables n = 2, nnz(P) = 2
equality constraints p = 1, nnz(A) = 2
inequality constraints m = 2, nnz(G) = 3
inequality lower bounds n_h_l = 0
inequality upper bounds n_h_u = 2
variable lower bounds n_x_l = 1
variable upper bounds n_x_u = 1

iter  solved       gap_max     p_res_max     d_res_max     rho_max   delta_max      mu_max  p_step  d_step
   0     0/4   1.90819e+01   2.69408e+00   4.84870e+00   1.000e-06   1.000e-04   1.652e+00  0.0000  0.0000
   1     0/4   2.16919e+03   2.52134e+00   4.41792e+00   1.000e-07   1.000e-05   2.362e+01  0.0648  0.9268
   2     0/4   1.48750e+06   2.44501e+00   1.44267e+00   5.000e-08   5.000e-06   1.140e+03  0.1477  0.1575
   

The results are still cupy arrays, same as the `DenseSolver`.

In [7]:
print(type(solver_sparse.result.x))

<class 'cupy.ndarray'>


## Verify the solution consistency

Both backends should report the same per-problem status, and agree on the optimum for
every problem that solved.

In [8]:
dense_status  = [st.name for st in solver_dense.result.info.status]
sparse_status = [st.name for st in solver_sparse.result.info.status]

# the two backends should agree on every problem's status ...
assert dense_status == sparse_status

# ... and on the problems that solved, the dense and sparse optima agree.
solved = [i for i, st in enumerate(solver_dense.result.info.status)
          if st == Status.CUPIQP_SOLVED]
x_dense  = torch.from_dlpack(solver_dense.result.x)
x_sparse = torch.from_dlpack(solver_sparse.result.x)
assert torch.allclose(x_dense[solved], x_sparse[solved], atol=1e-6)

print("dense and sparse backends agree on every problem.")

dense and sparse backends agree on every problem.
